# My Name in History : Name Disambiguation Pipeline
**Datasets:** Parish Court Records · ENB Books · ENB Persons  
**Goal:** Extract names from 19th-century court records, match them against ENB persons, and produce a slim combined dataset ready for the heritage product.


## 1 · Install dependencies

In [ ]:
!pip install rapidfuzz jellyfish beautifulsoup4 tqdm gdown --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 22.0 MB/s eta 0:00:00


## 2 · Mount Google Drive (for saving outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


## 3 · Download datasets from shared Drive links


In [ ]:
import gdown
import os

# shareable links
COURTS_LINK  = 'https://drive.google.com/file/d/1IYPzrSf1DVum_kOgG_fmpch-qyx8qDql/view?usp=sharing'
ENB_LINK     = 'https://drive.google.com/file/d/1Irau47A1yVw3aTfTEJJIBYf5icHUwGiS/view?usp=sharing'
PERSONS_LINK = 'https://drive.google.com/file/d/1gPThjUqqxtZqyNPkYIgjQUQnuxnno6Z0/view?usp=sharing'

# Download within Colab session storage
print('Downloading persons file...')
gdown.download(PERSONS_LINK, '/content/persons.tsv', fuzzy=True, quiet=False)

print('Downloading ENB books file...')
gdown.download(ENB_LINK, '/content/enb_books.tsv', fuzzy=True, quiet=False)

print('Downloading court records file (large, may take a few minutes)...')
gdown.download(COURTS_LINK, '/content/parish_courts_full_dataset.csv', fuzzy=True, quiet=False)

# Verify downloaded files
for f in ['/content/persons.tsv', '/content/enb_books.tsv',
          '/content/parish_courts_full_dataset.csv']:
    size_mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f'{os.path.exists(f)} | {size_mb:.1f} MB | {f}')


Downloading...
From: https://drive.google.com/uc?id=1gPThjUqqxtZqyNPkYIgjQUQnuxnno6Z0
To: /content/persons.tsv
100%|██████████| 21.7M/21.7M [00:00<00:00, 41.7MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1Irau47A1yVw3aTfTEJJIBYf5icHUwGiS
From (redirected): https://drive.google.com/uc?id=1Irau47A1yVw3aTfTEJJIBYf5icHUwGiS&confirm=t&uuid=51f857c9-2e29-4749-a6d8-503d29331bdc
To: /content/enb_books.tsv
100%|██████████| 194M/194M [00:03<00:00, 60.8MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1IYPzrSf1DVum_kOgG_fmpch-qyx8qDql
From (redirected): https://drive.google.com/uc?id=1IYPzrSf1DVum_kOgG_fmpch-qyx8qDql&confirm=t&uuid=773dc3b9-3c60-439f-9b64-9e90e233015e
To: /content/parish_courts_full_dataset.csv
100%|██████████| 398M/398M [00:04<00:00, 89.3MB/s]

True | 21.7 MB | /content/persons.tsv
True | 193.6 MB | /content/enb_books.tsv
True | 398.3 MB | /content/parish_courts_full_dataset.csv


## 4 ·  file paths

In [ ]:
import os

# Input files — downloaded into Colab session storage
COURTS_FILE  = '/content/parish_courts_full_dataset.csv'
ENB_FILE     = '/content/enb_books.tsv'
PERSONS_FILE = '/content/persons.tsv'

# Output — saved to Drive so results persist after session ends
OUTPUT_PATH  = '/content/drive/MyDrive/estonian_heritage/hum_hack/outputs/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

print('All paths set.')
for f in [COURTS_FILE, ENB_FILE, PERSONS_FILE]:
    print(os.path.exists(f), '|', f)
print('Output path ready:', OUTPUT_PATH)


All paths set.
True | /content/parish_courts_full_dataset.csv
True | /content/enb_books.tsv
True | /content/persons.tsv
Output path ready: /content/drive/MyDrive/estonian_heritage/hum_hack/outputs/


## 5 · Imports

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup
from rapidfuzz import fuzz, process
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Imports ready.')


Imports ready.


## 6 · Loading datasets

In [ ]:
# ENB Persons (small, load fully)
persons = pd.read_csv(PERSONS_FILE, sep='\t', encoding='utf-8')
print(f'Persons loaded: {len(persons):,} rows')
print(f'Columns: {list(persons.columns)}')


Persons loaded: 117,964 rows
Columns: ['id', 'creator', 'name', 'birth_date', 'death_date', 'profession', 'gender', 'name_varform', 'geographic_iso', 'biographical_info', 'viaf_id', 'wkp_id']


In [ ]:
#  ENB Books (load key columns only)
ENB_COLS = ['id', 'creator', 'contributor', 'publication_date_cleaned',
            'publication_place_harmonized', 'publication_place_latitude',
            'publication_place_longitude', 'language', 'topic_keyword',
            'genre_keyword', 'title', 'publisher_harmonized',
            'publication_decade', 'is_fiction', 'copyright_status']

erb = pd.read_csv(ENB_FILE, sep='\t', encoding='utf-8', usecols=ENB_COLS)
print(f'ENB books loaded: {len(erb):,} rows')
print(f'Columns: {list(erb.columns)}')


ENB books loaded: 317,154 rows
Columns: ['id', 'creator', 'contributor', 'title', 'publisher_harmonized', 'publication_date_cleaned', 'publication_decade', 'publication_place_harmonized', 'publication_place_latitude', 'publication_place_longitude', 'language', 'is_fiction', 'topic_keyword', 'genre_keyword', 'copyright_status']


In [ ]:
# Parish Courts (large, chunked load, key columns only)
COURT_COLS = ['id', 'year', 'month', 'day', 'text', 'jury',
              'maakond', 'kihelkond', 'vald', '_record_type']

print('Loading court records in chunks (this takes a few minutes)...')
chunks = []
for chunk in tqdm(pd.read_csv(COURTS_FILE, sep='|', encoding='utf-8',
                               usecols=COURT_COLS, chunksize=200_000,
                               low_memory=False)):
    chunks.append(chunk)
courts = pd.concat(chunks, ignore_index=True)
print(f'Court records loaded: {len(courts):,} rows')


Loading court records in chunks (this takes a few minutes)...


2it [00:07,  3.84s/it]

Court records loaded: 203,627 rows


## 7 · Name normalization
Two layers: German→Estonian first name lookup, and Estonian-aware phonetic normalization.


In [ ]:
FIRSTNAME_MAP = {
    'Johann': 'Juhan', 'Johannes': 'Juhan', 'Juhhan': 'Juhan', 'Johan': 'Juhan',
    'Hans':   'Jaan',  'Hanns': 'Jaan', 'Hannes': 'Jaan',
    'Jacob':  'Jaak',  'Jakob': 'Jaak',
    'Peter':  'Peeter','Petter': 'Peeter',
    'Michael':'Mihkel','Michel': 'Mihkel', 'Michkel': 'Mihkel',
    'Andreas':'Andres','Anders': 'Andres',
    'Thomas': 'Toomas','Tomas':  'Toomas',
    'Katharina': 'Kadri', 'Catharina': 'Kadri', 'Katarina': 'Kadri',
    'Elisabeth': 'Liis', 'Elizabeth': 'Liis', 'Lisa': 'Liis',
    'Jürri':  'Jüri',  'Jurri':  'Jüri',
    'Tönno':  'Tõnu',  'Tonno':  'Tõnu', 'Tõnno': 'Tõnu', 'Tönnis': 'Tõnis',
    'Maddis': 'Madis', 'Mattis': 'Madis',
    'Kaarl':  'Karl',  'Kaarel': 'Karl',
    'Friedrich': 'Priidu', 'Fridrich': 'Priidu',
    'Georg':  'Jüri',  'George': 'Jüri',
    'Heinrich': 'Hendrik', 'Henrich': 'Hendrik',
    'Nicolaus': 'Nikolai', 'Nikolaus': 'Nikolai',
    'Willhelm': 'Vilhelm', 'Wilhelm': 'Vilhelm',
    'Lisbeth': 'Liisbet', 'Lisbet': 'Liisbet',
    'Maria':   'Mari',  'Marie': 'Mari',
    'Anna':    'Anne',
    'Margaretha': 'Margareeta', 'Margarete': 'Margareeta',
}

def normalize_name(name):
    if not name or pd.isna(name):
        return ''
    parts = str(name).strip().split()
    if not parts:
        return ''
    parts[0] = FIRSTNAME_MAP.get(parts[0], parts[0])
    return ' '.join(parts)

def phonetic_normalize(name):
    if not name:
        return ''
    s = name.lower()
    for old, new in [
        ('jj','j'),('kk','k'),('ll','l'),('mm','m'),('nn','n'),
        ('pp','p'),('rr','r'),('ss','s'),('tt','t'),
        ('ph','f'),('ck','k'),('cz','ts'),
        ('w','v'),('y','i'),('ö','õ'),
        ('ae','ä'),('oe','õ'),('ue','ü'),('gg','k'),
    ]:
        s = s.replace(old, new)
    return s

print('Normalization functions ready.')


Normalization functions ready.


## 8 · Extracting names from court records

In [ ]:
def extract_names_from_text(html_text):
    if pd.isna(html_text):
        return []
    soup = BeautifulSoup(html_text, 'html.parser')
    names = []
    for tag in soup.find_all('person'):
        title = tag.get('title', '').strip()
        if not title or title.startswith('Koht:'):
            continue
        if title.startswith('Isik:'):
            title = title[5:].strip()
        if title:
            names.append((title, 'text'))
    return names

def parse_jury_field(jury_str):
    if pd.isna(jury_str):
        return []
    results = []
    for entry in str(jury_str).split(';'):
        entry = entry.strip()
        if not entry:
            continue
        variant_match = re.search(r'_([^_]+)_', entry)
        if variant_match:
            name = variant_match.group(1).strip()
        else:
            parts = entry.rsplit('-', 1)
            name = parts[0].strip()
        role_match = re.search(r'-([A-ZÄÖÜÕa-zäöüõ]+)$', entry)
        role = role_match.group(1) if role_match else 'Unknown'
        if name and len(name) > 1:
            results.append((name, role, 'jury'))
    return results

print('Extraction functions ready.')


Extraction functions ready.


In [ ]:
print('Extracting names from court records (this takes several minutes)...')
all_name_records = []

for _, row in tqdm(courts.iterrows(), total=len(courts)):
    record_id   = row['id']
    year        = row['year']
    maakond     = row['maakond']
    kihelkond   = row['kihelkond']
    vald        = row['vald']
    record_type = row['_record_type']

    for name, source in extract_names_from_text(row['text']):
        all_name_records.append({
            'court_record_id': record_id,
            'court_year':      year,
            'maakond':         maakond,
            'kihelkond':       kihelkond,
            'vald':            vald,
            'record_type':     record_type,
            'extracted_name':  name,
            'name_role':       'litigant_or_mentioned',
            'name_source':     source,
        })

    for name, role, source in parse_jury_field(row['jury']):
        all_name_records.append({
            'court_record_id': record_id,
            'court_year':      year,
            'maakond':         maakond,
            'kihelkond':       kihelkond,
            'vald':            vald,
            'record_type':     record_type,
            'extracted_name':  name,
            'name_role':       role,
            'name_source':     source,
        })

names_df = pd.DataFrame(all_name_records)
names_df['name_normalized'] = names_df['extracted_name'].apply(normalize_name)
names_df['name_phonetic']   = names_df['name_normalized'].apply(phonetic_normalize)

print(f'Total name occurrences extracted: {len(names_df):,}')
print(f'Unique names: {names_df["extracted_name"].nunique():,}')


Extracting names from court records (this takes several minutes)...


100%|██████████| 203627/203627 [05:22<00:00, 632.09it/s]


Total name occurrences extracted: 1,606,714
Unique names: 287,442


## 9 · Building ENB persons index

In [ ]:
persons['name_normalized']        = persons['name'].apply(normalize_name)
persons['name_phonetic']          = persons['name_normalized'].apply(phonetic_normalize)
persons['name_varform_normalized']= persons['name_varform'].apply(
    lambda x: normalize_name(x) if pd.notna(x) else '')

persons['birth_date'] = pd.to_numeric(persons['birth_date'], errors='coerce')
persons['death_date'] = pd.to_numeric(
    persons['death_date'].astype(str).str[:4], errors='coerce')

# Prioritized subset: Estonian persons active 1750–1950
persons_estonian = persons[
    (persons['geographic_iso'] == 'ee') |
    ((persons['birth_date'] >= 1750) & (persons['birth_date'] <= 1950))
].copy().reset_index(drop=True)

print(f'Full persons index:   {len(persons):,}')
print(f'Estonian-era subset:  {len(persons_estonian):,}')

ALL_NAMES      = persons['name_normalized'].tolist()
ESTONIAN_NAMES = persons_estonian['name_normalized'].tolist()


Full persons index:   117,964
Estonian-era subset:  82,406


## 10 · Aggregate publication statistics per person

In [ ]:
def iter_person_names(row):
    pub_date = row.get('publication_date_cleaned', None)
    book_id  = row.get('id', None)
    for field in ['creator', 'contributor']:
        val = row.get(field, '')
        if pd.isna(val):
            continue
        for entry in str(val).split(';'):
            m = re.match(r'^([^(\[]+)', entry.strip())
            if m:
                yield m.group(1).strip(), pub_date, book_id

pub_records = []
for _, row in tqdm(erb.iterrows(), total=len(erb), desc='Parsing ENB books'):
    for name, pub_date, book_id in iter_person_names(row):
        pub_records.append({'name': name, 'pub_year': pub_date, 'book_id': book_id})

pub_df = pd.DataFrame(pub_records)
pub_stats = pub_df.groupby('name').agg(
    publication_count      = ('book_id', 'count'),
    first_publication_year = ('pub_year', 'min'),
    last_publication_year  = ('pub_year', 'max'),
).reset_index()

pub_stats['name_normalized'] = pub_stats['name'].apply(normalize_name)
print(f'Unique person entries in ENB books: {len(pub_stats):,}')


Parsing ENB books: 100%|██████████| 317154/317154 [00:26<00:00, 11930.04it/s]


Unique person entries in ENB books: 140,792


## 11 · Disambiguation engine

In [ ]:
def era_plausible(court_year, birth_year, death_year):
    try:
        cy = int(court_year)
        if pd.notna(birth_year):
            by = int(birth_year)
            if cy < by + 10 or cy > by + 90:
                return False
        if pd.notna(death_year):
            dy = int(death_year)
            if cy > dy + 5:
                return False
    except (TypeError, ValueError):
        pass
    return True

def score_candidate(query_norm, query_phon, candidate_row, court_year):
    c_norm = candidate_row['name_normalized']
    c_phon = candidate_row['name_phonetic']
    c_var  = candidate_row['name_varform_normalized']
    birth  = candidate_row.get('birth_date')
    death  = candidate_row.get('death_date')

    score = 0.0

    if query_norm.lower() == c_norm.lower():
        score += 0.50

    score += (fuzz.token_sort_ratio(query_norm, c_norm) / 100.0) * 0.25

    if query_phon and c_phon:
        score += (fuzz.ratio(query_phon, c_phon) / 100.0) * 0.15

    if c_var:
        score += (fuzz.token_sort_ratio(query_norm, c_var) / 100.0) * 0.05

    if candidate_row.get('geographic_iso') == 'ee':
        score += 0.05

    if not era_plausible(court_year, birth, death):
        score *= 0.3

    return round(min(score, 1.0), 4)

def disambiguate(name_query, court_year=None, top_n=3, prefer_estonian=True):
    if not name_query or pd.isna(name_query):
        return []

    q_norm = normalize_name(str(name_query))
    q_phon = phonetic_normalize(q_norm)

    name_pool   = ESTONIAN_NAMES if prefer_estonian else ALL_NAMES
    person_pool = persons_estonian if prefer_estonian else persons

    candidates_raw = process.extract(
        q_norm, name_pool, scorer=fuzz.token_sort_ratio, limit=20)

    results = []
    seen = set()
    for _, _, idx in candidates_raw:
        row = person_pool.iloc[idx]
        pid = row['id']
        if pid in seen:
            continue
        seen.add(pid)
        conf = score_candidate(q_norm, q_phon, row, court_year)
        results.append({
            'enb_person_id':      pid,
            'enb_name':           row['name'],
            'enb_birth_date':     row.get('birth_date'),
            'enb_death_date':     row.get('death_date'),
            'enb_gender':         row.get('gender'),
            'enb_profession':     row.get('profession'),
            'enb_geographic_iso': row.get('geographic_iso'),
            'enb_name_varform':   row.get('name_varform'),
            'confidence':         conf,
        })

    results.sort(key=lambda x: x['confidence'], reverse=True)
    return results[:top_n]

print('Disambiguation engine ready.')
print()
for test_name, test_year in [('Jüri Soots', 1885), ('Johann Moks', 1870), ('Andres Kukk', 1875)]:
    matches = disambiguate(test_name, test_year, top_n=2)
    print(f'  {test_name} ({test_year}) ->', [(m["enb_name"], m["confidence"]) for m in matches])


Disambiguation engine ready.

  Jüri Soots (1885) -> [('Roots, Jüri', 0.1022), ('Soontak, Jüri', 0.0983)]
  Johann Moks (1870) -> [('Maaker, Juhan', 0.3058), ('Johann', 0.2827)]
  Andres Kukk (1875) -> [('Mikk, Andres', 0.1117), ('Koks, Andres', 0.1105)]


## 12 · Runing disambiguation across all unique court names

In [ ]:
names_df['decade'] = (names_df['court_year'] // 10 * 10).astype('Int64')
unique_queries = names_df[['name_normalized','decade']].drop_duplicates().copy()
print(f'Unique (name, decade) queries: {len(unique_queries):,}')

match_records = []
for _, row in tqdm(unique_queries.iterrows(), total=len(unique_queries), desc='Disambiguating'):
    name   = row['name_normalized']
    decade = row['decade']
    year   = int(decade) + 5 if pd.notna(decade) else None

    matches = disambiguate(name, court_year=year, top_n=1)
    if matches:
        best = matches[0]
        match_records.append({
            'name_normalized':    name,
            'decade':             decade,
            'enb_person_id':      best['enb_person_id'],
            'enb_name':           best['enb_name'],
            'enb_birth_date':     best['enb_birth_date'],
            'enb_death_date':     best['enb_death_date'],
            'enb_gender':         best['enb_gender'],
            'enb_profession':     best['enb_profession'],
            'enb_geographic_iso': best['enb_geographic_iso'],
            'enb_name_varform':   best['enb_name_varform'],
            'match_confidence':   best['confidence'],
        })
    else:
        match_records.append({'name_normalized': name, 'decade': decade})

matches_df = pd.DataFrame(match_records)
high_conf = (matches_df['match_confidence'] > 0.6).sum()
print(f'Done. High-confidence matches (>0.6): {high_conf:,}')


Unique (name, decade) queries: 321,480


Disambiguating: 100%|██████████| 321480/321480 [4:07:48<00:00, 21.62it/s]


Done. High-confidence matches (>0.6): 80


## 13 · Output 1 : Name lookup table

In [ ]:
lookup_table = (
    names_df[['extracted_name','name_normalized','name_phonetic']]
    .drop_duplicates(subset='name_normalized')
    .merge(matches_df.drop(columns=['decade']), on='name_normalized', how='left')
    .merge(pub_stats[['name_normalized','publication_count',
                       'first_publication_year','last_publication_year']],
           on='name_normalized', how='left')
)

lookup_table.to_csv(OUTPUT_PATH + 'name_lookup_table.csv', index=False, encoding='utf-8')
print(f'Lookup table saved: {len(lookup_table):,} unique names')
print(lookup_table[lookup_table['match_confidence'] > 0.7].head(10).to_string())


Lookup table saved: 321,485 unique names
     extracted_name name_normalized name_phonetic enb_person_id  enb_name  enb_birth_date  enb_death_date enb_gender enb_profession enb_geographic_iso                                enb_name_varform  match_confidence  publication_count  first_publication_year  last_publication_year
1272         Johann           Juhan         juhan     a11435045    Johann          1801.0          1873.0       male            NaN                NaN  Philalethes; Johann Nepomuk Maria Joseph Anton            0.9078                5.0                  1514.0                 2019.0
1273         Johann           Juhan         juhan     a11435045    Johann          1801.0          1873.0       male            NaN                NaN  Philalethes; Johann Nepomuk Maria Joseph Anton            0.9078                5.0                  1514.0                 2019.0
1275         Johann           Juhan         juhan     a11331215  Johannes          1881.0          1963.0     

## 14 · Output 2 : Enriched court records

In [ ]:
enriched = names_df.merge(
    matches_df.drop(columns=['decade']), on='name_normalized', how='left'
).merge(
    pub_stats[['name_normalized','publication_count',
               'first_publication_year','last_publication_year']],
    on='name_normalized', how='left'
)

enriched.to_csv(OUTPUT_PATH + 'court_records_enriched.csv', index=False, encoding='utf-8')
print(f'Enriched dataset saved: {len(enriched):,} rows')


Enriched dataset saved: 4,097,046 rows


## 15 · Output 3 : Slim combined dataset

In [ ]:
SLIM_COLS = [
    'court_record_id', 'court_year', 'maakond', 'kihelkond', 'vald',
    'record_type', 'extracted_name', 'name_normalized', 'name_role', 'name_source',
    'enb_person_id', 'enb_name', 'enb_birth_date', 'enb_death_date',
    'enb_gender', 'enb_profession', 'enb_geographic_iso', 'enb_name_varform',
    'publication_count', 'first_publication_year', 'last_publication_year',
    'match_confidence',
]

slim = enriched[[c for c in SLIM_COLS if c in enriched.columns]].copy()
slim = slim[slim['extracted_name'].notna()]

slim.to_csv(OUTPUT_PATH + 'slim_combined_dataset.csv', index=False, encoding='utf-8')

import os
size_mb = os.path.getsize(OUTPUT_PATH + 'slim_combined_dataset.csv') / 1e6
print(f'Slim dataset saved: {len(slim):,} rows, {size_mb:.1f} MB')
print(f'Columns: {list(slim.columns)}')


Slim dataset saved: 4,097,046 rows, 822.0 MB
Columns: ['court_record_id', 'court_year', 'maakond', 'kihelkond', 'vald', 'record_type', 'extracted_name', 'name_normalized', 'name_role', 'name_source', 'enb_person_id', 'enb_name', 'enb_birth_date', 'enb_death_date', 'enb_gender', 'enb_profession', 'enb_geographic_iso', 'enb_name_varform', 'publication_count', 'first_publication_year', 'last_publication_year', 'match_confidence']


## 16 · Summary statistics

In [ ]:
print('PIPELINE SUMMARY ')
print(f'Court records processed:              {len(courts):,}')
print(f'Name occurrences extracted:           {len(names_df):,}')
print(f'Unique names:                         {names_df["name_normalized"].nunique():,}')
print(f'High-confidence matches (>0.7):       {(slim["match_confidence"] > 0.7).sum():,}')
print(f'Medium-confidence matches (0.5-0.7):  '
      f'{((slim["match_confidence"] >= 0.5) & (slim["match_confidence"] <= 0.7)).sum():,}')
print()
print(' TOP 10 MOST FREQUENT COURT NAMES ')
print(names_df['name_normalized'].value_counts().head(10))
print()
print(' RECORDS BY COUNTY')
print(slim['maakond'].value_counts().head(10))


=== PIPELINE SUMMARY ===
Court records processed:              203,627
Name occurrences extracted:           1,606,714
Unique names:                         269,009
High-confidence matches (>0.7):       56,622
Medium-confidence matches (0.5-0.7):  0

=== TOP 10 MOST FREQUENT COURT NAMES ===
name_normalized
Jaan             9659
Jüri             6159
Juhan            3817
Mihkel           3381
Jaan Mollok      2813
Jaan Lossmann    2704
F. Thal          2603
Mart             2534
Ado              2044
Jaan Perler      1999
Name: count, dtype: int64

=== RECORDS BY COUNTY ===
maakond
Tartu       962695
Harju       814267
Viljandi    718435
Lääne       475025
Saare       333759
Võru        265717
Pärnu       223953
Viru        138838
Volmari      99049
Järva        65308
Name: count, dtype: int64


## 17 · Interactive name lookup

In [ ]:
def lookup_name(name_input, top_n=5):
    print(f'\n=== Results for: "{name_input}" ===')
    print(f'Normalized: {normalize_name(name_input)}')
    print()

    court_hits = names_df[
        names_df['name_normalized'].str.lower() == normalize_name(name_input).lower()
    ]
    if len(court_hits):
        print(f'Court record appearances: {len(court_hits):,}')
        print('  Counties:  ', court_hits['maakond'].value_counts().head(3).to_dict())
        print('  Year range:', int(court_hits['court_year'].min()),
              '-', int(court_hits['court_year'].max()))
        print('  Roles:     ', court_hits['name_role'].value_counts().head(3).to_dict())
    else:
        print('Not found in court records.')
    print()

    year_hint = int(court_hits['court_year'].median()) if len(court_hits) else None
    matches = disambiguate(name_input, court_year=year_hint, top_n=top_n)
    if matches:
        print('Top ENB person matches:')
        for i, m in enumerate(matches, 1):
            pub_row = pub_stats[pub_stats['name_normalized'] == m['enb_name']]
            pub_count = int(pub_row['publication_count'].iloc[0]) if len(pub_row) else 0
            birth = int(m['enb_birth_date']) if pd.notna(m.get('enb_birth_date')) else '?'
            print(f'  {i}. {m["enb_name"]} (b.{birth})'
                  f' | {m["enb_profession"]}'
                  f' | confidence: {m["confidence"]}'
                  f' | publications: {pub_count}')
    else:
        print('No ENB matches found.')

# Try some example names
lookup_name('Jüri Soots')
lookup_name('Andres Kukk')



=== Results for: "Jüri Soots" ===
Normalized: Jüri Soots

Court record appearances: 3
  Counties:   {'Pärnu': 2, 'Viljandi': 1}
  Year range: 1876 - 1885
  Roles:      {'litigant_or_mentioned': 3}

Top ENB person matches:
  1. Roots, Jüri (b.1935) | nan | confidence: 0.1022 | publications: 2
  2. Soontak, Jüri (b.1901) | nan | confidence: 0.0983 | publications: 2
  3. Soomets, Arvi (b.1937) | nan | confidence: 0.0976 | publications: 2
  4. Kotkas, Jüri (b.1941) | nan | confidence: 0.0973 | publications: 2
  5. Koks, Jüri (b.1931) | koolipsühholoog | confidence: 0.0966 | publications: 1

=== Results for: "Andres Kukk" ===
Normalized: Andres Kukk

Court record appearances: 4
  Counties:   {'Viljandi': 4}
  Year range: 1852 - 1869
  Roles:      {'Peakohtumees': 2, 'litigant_or_mentioned': 2}

Top ENB person matches:
  1. Mikk, Andres (b.1984) | trummar | confidence: 0.1117 | publications: 0
  2. Koks, Andres (b.1964) | nan | confidence: 0.1105 | publications: 0
  3. Kaju, Andres (b.1963)

In [ ]:
!pip install gradio plotly folium requests --quiet
print('Frontend dependencies ready.')

Frontend dependencies ready.


In [ ]:
import pandas as pd
import os

OUTPUT_PATH = '/content/drive/MyDrive/estonian_heritage/hum_hack/outputs/'
SLIM_FILE   = OUTPUT_PATH + 'slim_combined_dataset.csv'

slim = pd.read_csv(SLIM_FILE, encoding='utf-8', low_memory=False)
print(f'Slim dataset loaded: {len(slim):,} rows')
print(f'Columns: {list(slim.columns)}')


Slim dataset loaded: 4,097,046 rows
Columns: ['court_record_id', 'court_year', 'maakond', 'kihelkond', 'vald', 'record_type', 'extracted_name', 'name_normalized', 'name_role', 'name_source', 'enb_person_id', 'enb_name', 'enb_birth_date', 'enb_death_date', 'enb_gender', 'enb_profession', 'enb_geographic_iso', 'enb_name_varform', 'publication_count', 'first_publication_year', 'last_publication_year', 'match_confidence']


In [ ]:
import requests
import time
import json

WIKIDATA_CACHE_FILE = OUTPUT_PATH + 'wikidata_enrichment.csv'

def fetch_wikidata_batch(qids, retries=3):
    """Fetch enrichment data for a batch of Wikidata QIDs via SPARQL."""
    ids_str = ' '.join(f'wd:{qid}' for qid in qids)
    query = f'''
    SELECT ?person ?personLabel ?birthDate ?deathDate
           ?birthPlace ?birthPlaceLabel ?birthPlaceLat ?birthPlaceLon
           ?image ?sitelinks
    WHERE {{
      VALUES ?person {{ {ids_str} }}
      OPTIONAL {{ ?person wdt:P569 ?birthDate }}
      OPTIONAL {{ ?person wdt:P570 ?deathDate }}
      OPTIONAL {{ ?person wdt:P19 ?birthPlace }}
      OPTIONAL {{
        ?person wdt:P19 ?bp .
        ?bp wdt:P625 ?coord .
        BIND(geof:latitude(?coord)  AS ?birthPlaceLat)
        BIND(geof:longitude(?coord) AS ?birthPlaceLon)
      }}
      OPTIONAL {{ ?person wdt:P18 ?image }}
      OPTIONAL {{ ?person wikibase:sitelinks ?sitelinks }}
      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "et,en"
      }}
    }}
    '''
    headers = {
        'User-Agent': 'EstonianHeritageHackathon/1.0 (hackathon; contact@example.com)',
        'Accept': 'application/sparql-results+json'
    }
    for attempt in range(retries):
        try:
            r = requests.get(
                'https://query.wikidata.org/sparql',
                params={'query': query, 'format': 'json'},
                headers=headers,
                timeout=30
            )
            if r.status_code == 200:
                return r.json()['results']['bindings']
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(2 ** attempt)
    return []

def parse_wikidata_results(raw_results):
    """Parse SPARQL results into a clean dict keyed by QID."""
    parsed = {}
    for row in raw_results:
        qid = row['person']['value'].split('/')[-1]
        if qid not in parsed:
            parsed[qid] = {
                'wkp_id':          qid,
                'wd_label':        row.get('personLabel', {}).get('value'),
                'wd_birth_date':   row.get('birthDate',   {}).get('value'),
                'wd_death_date':   row.get('deathDate',   {}).get('value'),
                'wd_birthplace':   row.get('birthPlaceLabel', {}).get('value'),
                'wd_birth_lat':    row.get('birthPlaceLat',   {}).get('value'),
                'wd_birth_lon':    row.get('birthPlaceLon',   {}).get('value'),
                'wd_image_url':    row.get('image',       {}).get('value'),
                'wd_sitelinks':    row.get('sitelinks',   {}).get('value'),
            }
        # Keep the image if we get one in a later row
        if 'image' in row and not parsed[qid]['wd_image_url']:
            parsed[qid]['wd_image_url'] = row['image']['value']
    return parsed

# Get unique QIDs from slim dataset that have a match
print('Collecting Wikidata IDs from slim dataset...')

# Load persons to get wkp_id mapping
PERSONS_FILE = '/content/persons.tsv'
persons_df = pd.read_csv(PERSONS_FILE, sep='\t', encoding='utf-8',
                         usecols=['id','wkp_id'])
persons_df = persons_df[persons_df['wkp_id'].notna()]

# Join to slim to find which matched persons have wkp_ids
matched_ids = slim[slim['enb_person_id'].notna()]['enb_person_id'].unique()
relevant_persons = persons_df[persons_df['id'].isin(matched_ids)]
qids = relevant_persons['wkp_id'].dropna().unique().tolist()
print(f'Unique QIDs to fetch: {len(qids):,}')

# Batch fetch : Wikidata SPARQL handles ~50 IDs per request safely
BATCH_SIZE = 50
all_parsed = {}

for i in range(0, len(qids), BATCH_SIZE):
    batch = qids[i:i+BATCH_SIZE]
    raw   = fetch_wikidata_batch(batch)
    parsed = parse_wikidata_results(raw)
    all_parsed.update(parsed)
    if (i // BATCH_SIZE) % 10 == 0:
        print(f'  Fetched {min(i+BATCH_SIZE, len(qids))}/{len(qids)}...')
    time.sleep(0.5)  # be polite to Wikidata

wikidata_df = pd.DataFrame(list(all_parsed.values()))
wikidata_df.to_csv(WIKIDATA_CACHE_FILE, index=False, encoding='utf-8')
print(f'Wikidata enrichment saved: {len(wikidata_df):,} persons')
print(f'With portrait image: {wikidata_df["wd_image_url"].notna().sum():,}')
print(f'With birthplace coords: {wikidata_df["wd_birth_lat"].notna().sum():,}')


Unique QIDs to fetch: 9,774
  Fetched 50/9774...
  Fetched 550/9774...
  Fetched 1050/9774...
  Fetched 1550/9774...
  Fetched 2050/9774...
  Fetched 2550/9774...
  Fetched 3050/9774...
  Fetched 3550/9774...
  Fetched 4050/9774...
  Fetched 4550/9774...
  Fetched 5050/9774...
  Fetched 5550/9774...
  Fetched 6050/9774...
  Fetched 6550/9774...
  Fetched 7050/9774...
  Fetched 7550/9774...
  Fetched 8050/9774...
  Fetched 8550/9774...
  Fetched 9050/9774...
  Fetched 9550/9774...
Wikidata enrichment saved: 9,774 persons
With portrait image: 4,814
With birthplace coords: 7,542


In [ ]:
import pandas as pd
import numpy as np
import re
import random
import warnings
warnings.filterwarnings('ignore')

!pip install gradio plotly folium --quiet

OUTPUT_PATH  = '/content/drive/MyDrive/estonian_heritage/hum_hack/outputs/'
PERSONS_FILE = '/content/drive/MyDrive/estonian_heritage/hum_hack/persons.tsv'

slim = pd.read_csv(OUTPUT_PATH + 'slim_enriched.csv', encoding='utf-8', low_memory=False)
print(f'Loaded: {len(slim):,} rows, {len(slim.columns)} columns')

# Load persons to get biographical_info
persons_bio = pd.read_csv(
    PERSONS_FILE, sep='\t', encoding='utf-8',
    usecols=['id', 'biographical_info', 'profession']
)
persons_bio.columns = ['enb_person_id', 'biographical_info', 'enb_profession_full']
slim = slim.merge(persons_bio, on='enb_person_id', how='left')

# Clean string "nan" values throughout
def clean_val(v):
    if pd.isna(v): return None
    if str(v).strip().lower() == 'nan': return None
    return str(v).strip()

for col in slim.select_dtypes(include='object').columns:
    slim[col] = slim[col].apply(clean_val)

# Fix Wikidata image URLs to proper Wikimedia thumbnail format
def fix_image_url(url):
    if not url: return None
    url = str(url)
    if 'Special:FilePath' in url:
        filename = url.split('Special:FilePath/')[-1]
        return f'https://commons.wikimedia.org/wiki/Special:FilePath/{filename}?width=400'
    return url

slim['wd_image_url'] = slim['wd_image_url'].apply(fix_image_url)

# Load ENB books for topic/language/fiction detail per person
ENB_COLS = ['id', 'creator', 'contributor', 'language', 'topic_keyword',
            'genre_keyword', 'is_fiction', 'publication_decade']
erb = pd.read_csv(OUTPUT_PATH.replace('outputs/', '') + 'enb_books.tsv',
                  sep='\t', encoding='utf-8', usecols=ENB_COLS)

def iter_names_from_books(row):
    for field in ['creator', 'contributor']:
        val = row.get(field, '')
        if pd.isna(val) or str(val).strip().lower() == 'nan':
            continue
        for entry in str(val).split(';'):
            m = re.match(r'^([^(\[]+)', entry.strip())
            if m:
                yield m.group(1).strip()

pub_detail_records = []
for _, row in erb.iterrows():
    for name in iter_names_from_books(row):
        pub_detail_records.append({
            'enb_name':          name,
            'language':          row.get('language'),
            'topic_keyword':     row.get('topic_keyword'),
            'is_fiction':        row.get('is_fiction'),
            'publication_decade':row.get('publication_decade'),
        })

pub_detail_df = pd.DataFrame(pub_detail_records)

def get_pub_profile(enb_name):
    if not enb_name: return {}
    subset = pub_detail_df[pub_detail_df['enb_name'] == enb_name]
    if subset.empty: return {}
    langs    = subset['language'].dropna().value_counts().head(2).index.tolist()
    topics   = []
    for t in subset['topic_keyword'].dropna():
        topics += [x.strip() for x in str(t).split(';') if x.strip()]
    top_topics = pd.Series(topics).value_counts().head(3).index.tolist()
    fiction_count    = int((subset['is_fiction'] == True).sum())
    nonfiction_count = int((subset['is_fiction'] == False).sum())
    top_decade = subset['publication_decade'].dropna().value_counts().index[0] \
                 if subset['publication_decade'].notna().any() else None
    return {
        'languages':   langs,
        'topics':      top_topics,
        'fiction':     fiction_count,
        'nonfiction':  nonfiction_count,
        'top_decade':  top_decade,
    }

# Pre-compute good demo names
def get_good_names(df, min_conf=0.65, min_court=3, top_n=20):
    candidates = (
        df[df['match_confidence'].fillna(0) >= min_conf]
        .groupby('name_normalized')
        .agg(
            court_count    = ('court_record_id', 'count'),
            confidence     = ('match_confidence', 'max'),
            has_image      = ('wd_image_url', lambda x: x.notna().any()),
            has_coords     = ('wd_birth_lat', lambda x: x.notna().any()),
            pub_count      = ('publication_count', 'max'),
            extracted_name = ('extracted_name', 'first'),
        ).reset_index()
    )
    candidates = candidates[candidates['court_count'] >= min_court]
    # Extract surname only (last token) for Surprise Me
    candidates['surname'] = candidates['extracted_name'].apply(
        lambda x: str(x).split()[-1] if pd.notna(x) else x)
    candidates['score'] = (
        candidates['confidence'] * 0.4 +
        candidates['has_image'].astype(float) * 0.25 +
        candidates['has_coords'].astype(float) * 0.15 +
        (candidates['pub_count'].fillna(0) > 0).astype(float) * 0.2
    )
    return candidates.sort_values('score', ascending=False).head(top_n)

good_names    = get_good_names(slim)
EXAMPLE_NAMES = good_names['surname'].tolist()
print(f'Good demo names found: {len(EXAMPLE_NAMES)}')
print('Top surnames:', EXAMPLE_NAMES[:6])

Loaded: 4,097,046 rows, 34 columns
Good demo names found: 20
Top surnames: ['Aleksander', 'Aleksandra', 'Nikolai', 'Adam', 'Johann', 'Gustav']


#Gradio Frontend and Visualization



In [ ]:
import gradio as gr
import plotly.graph_objects as go
import folium

def fame_label(sitelinks):
    if not sitelinks: return ""
    try: n = int(float(sitelinks))
    except: return ""
    if n >= 50: return f"Internationally renowned — {n} Wikipedia editions"
    if n >= 20: return f"Widely known in Europe — {n} Wikipedia editions"
    if n >= 5:  return f"Notable Estonian figure — {n} Wikipedia editions"
    if n >= 1:  return f"Documented — {n} Wikipedia edition"
    return ""

def clean(v):
    if v is None: return None
    if pd.isna(v) if not isinstance(v, str) else False: return None
    if str(v).strip().lower() in ('nan', 'none', ''): return None
    return str(v).strip()

def get_case_label(record_type):
    if not clean(record_type): return "Unknown"
    s = re.sub(r"^\d+\.\s*", "", str(record_type))
    return s[:70] + "..." if len(s) > 70 else s

def format_date(date_str):
    if not clean(date_str): return None
    try:
        # Wikidata returns ISO format like 1847-03-14T00:00:00Z
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})', str(date_str))
        if m:
            y, mo, d = m.groups()
            months = ['Jan','Feb','Mar','Apr','May','Jun',
                      'Jul','Aug','Sep','Oct','Nov','Dec']
            return f"{int(d)} {months[int(mo)-1]} {y}"
        m2 = re.search(r'(\d{4})', str(date_str))
        return m2.group(1) if m2 else None
    except: return None

COUNTY_COORDS = {
    "Harju":      [59.35, 24.85], "Tartu":     [58.38, 26.72],
    "Viljandi":   [58.36, 25.59], "Pärnu":     [58.38, 24.50],
    "Lääne":      [58.95, 23.54], "Saare":     [58.48, 22.55],
    "Võru":       [57.84, 27.02], "Valga":     [57.78, 26.05],
    "Järva":      [58.85, 25.55], "Lääne-Viru":[59.30, 26.33],
    "Ida-Viru":   [59.35, 27.41], "Põlva":     [58.05, 27.07],
    "Rapla":      [58.99, 24.73], "Hiiu":      [58.92, 22.59],
    "Jõgeva":     [58.75, 26.40],
}

def search_name(name_input):
    if not name_input or not name_input.strip():
        return "", None, None, "", "", ""

    q = name_input.strip()

    # Word-boundary match to avoid partial hits (Lee != Leenik)
    pattern = r'(?<![a-zäöüõA-ZÄÖÜÕ])' + re.escape(q) + r'(?![a-zäöüõA-ZÄÖÜÕ])'
    mask = (
        slim["extracted_name"].str.contains(pattern, na=False, regex=True, case=False) |
        slim["name_normalized"].str.contains(pattern, na=False, regex=True, case=False)
    )
    hits = slim[mask].copy()

    best = None
    if not hits.empty and "match_confidence" in hits.columns:
        ranked = hits.dropna(subset=["enb_person_id"]) \
                     .sort_values("match_confidence", ascending=False)
        if len(ranked):
            best = ranked.iloc[0].to_dict()

    if hits.empty:
        return (f"No records found for **{q}**. "
                f"Try a common Estonian surname like Tamm, Sepp, Kukk, or Mägi.",
                None, None, "", "", "")

    # Core stats
    count      = len(hits)
    yr_min     = int(hits["court_year"].min())
    yr_max     = int(hits["court_year"].max())
    top_county = hits["maakond"].value_counts().index[0] \
                 if hits["maakond"].notna().any() else None
    top_parish = hits["kihelkond"].value_counts().index[0] \
                 if "kihelkond" in hits and hits["kihelkond"].notna().any() else None
    top_vald   = hits["vald"].value_counts().index[0] \
                 if "vald" in hits and hits["vald"].notna().any() else None

    # Role breakdown
    role_counts  = hits["name_role"].value_counts()
    role_summary = ", ".join([f"{role} ({cnt}x)"
                              for role, cnt in role_counts.head(3).items()])

    # Most active decade
    hits['decade'] = (hits['court_year'] // 10 * 10)
    top_decade = int(hits['decade'].value_counts().index[0]) \
                 if hits['decade'].notna().any() else None
    decade_str = f"{top_decade}s" if top_decade else None

    # Case type breakdown
    case_counts = hits["record_type"].value_counts() \
                  if "record_type" in hits else pd.Series(dtype=int)
    top_cases   = [(get_case_label(ct), int(n))
                   for ct, n in case_counts.head(3).items()]

    #  Build summary
    summary_lines = [f"## {q}", ""]

    # Court records section
    summary_lines += [
        "**Court Records**", "",
        f"This name appears **{count:,} times** across Estonian parish court records "
        f"spanning **{yr_min} to {yr_max}**.",
    ]
    if top_county:
        geo_str = f"Most records are concentrated in **{top_county} county**"
        if top_parish: geo_str += f", specifically **{top_parish} parish**"
        if top_vald:   geo_str += f" ({top_vald} municipality)"
        summary_lines.append(geo_str + ".")
    if decade_str:
        summary_lines.append(f"The name was most active in court records during the **{decade_str}**.")
    if role_summary:
        summary_lines.append(f"Roles in court: {role_summary}.")
    if top_cases:
        cases_str = "; ".join([f"{label} ({n}x)" for label, n in top_cases])
        summary_lines.append(f"Most common case types: {cases_str}.")

    image_url = None

    # ENB match section
    if best:
        enb_name    = clean(best.get("enb_name"))
        profession  = clean(best.get("enb_profession_full") or best.get("enb_profession"))
        bio         = clean(best.get("biographical_info"))
        birthplace  = clean(best.get("wd_birthplace"))
        birth_date  = format_date(best.get("wd_birth_date")) or \
                      clean(best.get("enb_birth_date"))
        death_date  = format_date(best.get("wd_death_date")) or \
                      clean(best.get("enb_death_date"))
        pub_count   = best.get("publication_count")
        first_pub   = best.get("first_publication_year")
        last_pub    = best.get("last_publication_year")
        sitelinks   = best.get("wd_sitelinks")
        conf        = best.get("match_confidence", 0)

        pub_profile = get_pub_profile(enb_name) if enb_name else {}

        summary_lines += ["", "---", "**National Bibliography Match**", ""]

        # Bio note if available
        if bio:
            summary_lines.append(f"_{bio}_")
            summary_lines.append("")

        # Name + dates
        identity = f"**{enb_name}**" if enb_name else "Unknown"
        if birth_date: identity += f" · born {birth_date}"
        if death_date: identity += f" · died {death_date}"
        summary_lines.append(identity)

        if profession:  summary_lines.append(f"Profession: {profession}")
        if birthplace:  summary_lines.append(f"Born in: {birthplace}")

        # Publications
        if pub_count and pd.notna(pub_count) and int(pub_count) > 0:
            pub_line = f"Published **{int(pub_count)} work(s)**"
            if first_pub and pd.notna(first_pub):
                pub_line += f" between {int(first_pub)}"
                if last_pub and pd.notna(last_pub) and int(last_pub) != int(first_pub):
                    pub_line += f" and {int(last_pub)}"
            if pub_profile.get('top_decade'):
                pub_line += f", most actively in the {pub_profile['top_decade']}s"
            summary_lines.append(pub_line + ".")

            if pub_profile.get('languages'):
                lang_str = ", ".join(pub_profile['languages'])
                summary_lines.append(f"Published in: {lang_str}.")

            if pub_profile.get('fiction') or pub_profile.get('nonfiction'):
                f_n  = pub_profile.get('fiction', 0)
                nf_n = pub_profile.get('nonfiction', 0)
                if f_n and nf_n:
                    summary_lines.append(f"Works: {f_n} fiction, {nf_n} non-fiction.")
                elif f_n:
                    summary_lines.append(f"Published fiction.")
                elif nf_n:
                    summary_lines.append(f"Published non-fiction.")

            if pub_profile.get('topics'):
                topics_str = ", ".join(pub_profile['topics'])
                summary_lines.append(f"Themes: {topics_str}.")

        fame = fame_label(sitelinks)
        if fame: summary_lines.append(f"\n{fame}")
        summary_lines.append(f"\nMatch confidence: **{float(conf):.0%}**")

        # Portrait
        img = clean(best.get("wd_image_url"))
        if img: image_url = img

    summary = "\n".join(summary_lines)

    # Timeline
    fig = go.Figure()
    year_counts = hits.groupby("court_year").size().reset_index(name="count")

    fig.add_trace(go.Bar(
        x=year_counts["court_year"],
        y=-year_counts["count"],
        marker_color="#c0392b",
        marker_opacity=0.75,
        name="Court appearances",
        hovertemplate="%{x}: %{customdata} records<extra></extra>",
        customdata=year_counts["count"],
    ))

    if best:
        first_pub = best.get("first_publication_year")
        last_pub  = best.get("last_publication_year")
        pub_count = best.get("publication_count", 0)
        if first_pub and pd.notna(first_pub) and pub_count and int(pub_count) > 0:
            pub_years = [int(first_pub)]
            if last_pub and pd.notna(last_pub) and int(last_pub) != int(first_pub):
                pub_years.append(int(last_pub))
            fig.add_trace(go.Scatter(
                x=pub_years,
                y=[max(year_counts["count"]) * 0.6] * len(pub_years),
                mode="markers+text",
                marker=dict(size=16, color="#2980b9", symbol="star",
                            line=dict(width=1, color="white")),
                text=["First publication", "Last publication"][:len(pub_years)],
                textposition="top center",
                textfont=dict(size=10, color="#2980b9"),
                name="ENB publications",
                hovertemplate="%{x}<extra></extra>",
            ))
            if yr_max < int(first_pub):
                fig.add_annotation(
                    x=(yr_max + int(first_pub)) / 2,
                    y=max(year_counts["count"]) * 0.3,
                    text=f"{int(first_pub) - yr_max} years later",
                    showarrow=False,
                    font=dict(size=10, color="#7f8c8d"),
                )

    fig.add_hline(y=0, line_color="#bdc3c7", line_width=1)
    fig.add_annotation(x=0, y=1, xref="paper", yref="paper",
                       text="Publications", showarrow=False,
                       font=dict(size=11, color="#2980b9"), xanchor="left")
    fig.add_annotation(x=0, y=0, xref="paper", yref="paper",
                       text="Court records", showarrow=False,
                       font=dict(size=11, color="#c0392b"),
                       xanchor="left", yanchor="top")
    fig.update_layout(
        title=dict(text=f"The arc of '{q}' across history",
                   font=dict(size=15, family="Georgia, serif")),
        xaxis=dict(title="Year", showgrid=True,
                   gridcolor="#f0f0f0", zeroline=False),
        yaxis=dict(visible=False),
        height=320, showlegend=False,
        plot_bgcolor="white", paper_bgcolor="white",
        margin=dict(l=20, r=20, t=50, b=40),
        font=dict(family="Georgia, serif"),
        bargap=0.1,
    )

    # Narrative
    parts = []
    parts.append(
        f"The name **{q}** surfaces {count:,} times in Estonian parish court records "
        f"between **{yr_min}** and **{yr_max}**"
        + (f", concentrated in **{top_county}**" if top_county else "")
        + (f", most actively during the **{decade_str}**" if decade_str else "")
        + "."
    )
    if role_counts is not None and len(role_counts):
        top_r = role_counts.index[0]
        parts.append(
            f"In these records the name appears most frequently as **{top_r}** — "
            f"a window into the social standing and legal life of this family in "
            f"19th-century rural Estonia."
        )
    if best:
        enb_name  = clean(best.get("enb_name"))
        first_pub = best.get("first_publication_year")
        pub_count = best.get("publication_count", 0)
        bio       = clean(best.get("biographical_info"))
        profession= clean(best.get("enb_profession_full") or best.get("enb_profession"))

        if pub_count and pd.notna(pub_count) and int(pub_count) > 0:
            t = f"By **{int(first_pub)}**, a {enb_name} had entered the Estonian National Bibliography"
            if int(pub_count) > 1: t += f" with **{int(pub_count)} published works**"
            if profession: t += f", working as a **{profession}**"
            parts.append(t + ".")

            if yr_max < int(first_pub):
                gap = int(first_pub) - yr_max
                parts.append(
                    f"In roughly **{gap} years**, this name crossed from the court ledger "
                    f"to the title page — from subject of documentation to author of culture. "
                    f"That crossing is the story of Estonia finding its written voice."
                )

        if pub_profile.get('topics'):
            parts.append(
                f"Their work touched on **{', '.join(pub_profile['topics'])}** — "
                f"themes that echo the concerns of the very community whose disputes "
                f"once filled the parish court records."
            )

        fame = fame_label(best.get("wd_sitelinks"))
        if fame:
            parts.append(f"Today, this name is remembered beyond Estonia's borders: **{fame}**.")

    narrative = "\n\n".join(parts) if parts else \
                f"**{q}** appears in court records but no confident ENB match was found."

    #  Map
    lat = clean(best.get("wd_birth_lat")) if best else None
    lon = clean(best.get("wd_birth_lon")) if best else None

    try:
        center = [float(lat), float(lon)] if lat and lon else [58.5953, 25.0136]
        zoom   = 8 if lat and lon else 6
    except:
        center, zoom = [58.5953, 25.0136], 6

    m = folium.Map(location=center, zoom_start=zoom, tiles="CartoDB positron")

    if lat and lon:
        try:
            folium.Marker(
                location=[float(lat), float(lon)],
                popup=folium.Popup(
                    f"<b>{clean(best.get('enb_name',''))}</b>"
                    f"<br>Born in {clean(best.get('wd_birthplace',''))}",
                    max_width=200),
                tooltip=f"Born in {clean(best.get('wd_birthplace',''))}",
                icon=folium.Icon(color="blue", icon="star", prefix="fa"),
            ).add_to(m)
        except: pass

    county_counts = hits["maakond"].value_counts()
    max_c = county_counts.max() if len(county_counts) else 1
    for county, cnt in county_counts.items():
        if county in COUNTY_COORDS:
            folium.CircleMarker(
                location=COUNTY_COORDS[county],
                radius=int(6 + (cnt / max_c) * 18),
                color="#c0392b", fill=True,
                fill_color="#c0392b", fill_opacity=0.35,
                popup=folium.Popup(
                    f"<b>{county}</b><br>{cnt} records", max_width=150),
                tooltip=f"{county}: {cnt} records",
            ).add_to(m)

    return summary, image_url, fig, narrative, m._repr_html_(), ""

#  UI

CSS = """
.gradio-container { font-family: Georgia, serif !important; max-width: 1100px !important; margin: auto !important; }
#title-block { background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); border-radius: 12px; padding: 32px 24px; margin-bottom: 8px; text-align: center; }
#title-block h1 { color: #f5e6c8 !important; font-size: 2.4em !important; letter-spacing: 0.02em; margin: 0 0 8px 0; }
#title-block p { color: #c9a96e !important; font-size: 1.05em; margin: 0; }
.summary-card { font-size: 0.93em !important; line-height: 1.75 !important; }
"""

with gr.Blocks(css=CSS, title="My Name in History") as demo:

    gr.HTML("""
    <div id="title-block">
      <h1>My Name in History</h1>
      <p>Trace your family name from 19th-century Estonian court records to published authorship</p>
    </div>
    """)

    with gr.Row():
        name_box     = gr.Textbox(
            label="Enter a family name",
            placeholder="e.g. Tamm, Sepp, Kukk, Mägi, Soots...",
            scale=5)
        search_btn   = gr.Button("Search", variant="primary", scale=1)
        surprise_btn = gr.Button("Surprise me", scale=1)

    gr.Examples(
        examples=[[n] for n in EXAMPLE_NAMES[:6]],
        inputs=name_box,
        label="Try these family names",
    )

    status_box = gr.Markdown(visible=False)

    with gr.Row():
        with gr.Column(scale=1, min_width=240):
            portrait_img = gr.Image(
                label="Portrait (Wikidata)",
                height=250,
                show_download_button=False,
            )
            summary_md = gr.Markdown(
                label="Summary",
                elem_classes="summary-card",
            )
        with gr.Column(scale=2):
            timeline_plot = gr.Plot(label="Historical arc")
            narrative_md  = gr.Markdown(label="The story of this name")

    map_display = gr.HTML(label="Map — court record counties and birthplace")

    outputs = [summary_md, portrait_img, timeline_plot,
               narrative_md, map_display, status_box]

    search_btn.click(fn=search_name, inputs=name_box, outputs=outputs)
    name_box.submit(fn=search_name, inputs=name_box, outputs=outputs)

    def surprise():
        return random.choice(EXAMPLE_NAMES)

    surprise_btn.click(fn=surprise, outputs=name_box).then(
        fn=search_name, inputs=name_box, outputs=outputs)

    gr.HTML("""
    <div style="text-align:center; padding:16px 0 4px 0; color:#aaa; font-size:0.85em;">
        Built at Hum Hackathon 2026 &nbsp;·&nbsp;
        Data: National Archives of Estonia &amp; National Library of Estonia
    </div>
    """)

demo.launch(share=True, debug=False, quiet=True)
print("\nDemo is live. Share the public URL above.")


* Running on public URL: https://9e95b47902abc01079.gradio.live



Demo is live. Share the public URL above.
